In [1]:
#| default_exp perf.compression
#| export

import numpy as np
import copy
from typing import List, Dict, Any, Tuple, Optional
import time

# Import from TinyTorch package (previous modules must be completed and exported)
from tinytorch.core.tensor import Tensor
from tinytorch.core.layers import Linear, Sequential
from tinytorch.core.activations import ReLU

# Constants for memory calculations
BYTES_PER_FLOAT32 = 4  # Standard float32 size in bytes
MB_TO_BYTES = 1024 * 1024  # Megabytes to bytes conversion

# Sequential provides model container with .layers and .parameters()

In [2]:
# Profile weight distribution to discover pruning opportunities
# Module 14 (Profiling) must be completed before Module 16
from tinytorch.perf.profiling import Profiler, analyze_weight_distribution

def show_weight_distribution_motivation():
    """Display weight distribution analysis - motivates compression techniques."""
    profiler = Profiler()

    # Create a model and analyze its weights
    model = Linear(512, 512)
    input_data = Tensor(np.random.randn(1, 512))

    # Profile basic characteristics
    profile = profiler.profile_forward_pass(model, input_data)

    print("🧪 Profiling Parameter Distribution:\n")
    print(f"   Total parameters: {profile['parameters']:,}")
    print(f"   Model memory: {profile['parameters'] * BYTES_PER_FLOAT32 / MB_TO_BYTES:.1f} MB (FP32)")

    # Analyze weight distribution
    weights = model.weight.data.flatten()
    abs_weights = np.abs(weights)

    print("\n   Weight Statistics:")
    print(f"   Mean: {np.mean(abs_weights):.4f}")
    print(f"   Std:  {np.std(abs_weights):.4f}")
    print(f"   Min:  {np.min(abs_weights):.4f}")
    print(f"   Max:  {np.max(abs_weights):.4f}")

    # Check how many weights are small
    thresholds = [0.001, 0.01, 0.1, 0.5]
    print("\n   Weights Below Threshold:")
    print("   Threshold  |  Percentage")
    print("   -----------|--------------")
    for threshold in thresholds:
        percentage = np.sum(abs_weights < threshold) / len(weights) * 100
        print(f"   < {threshold:<6}  |  {percentage:5.1f}%")

    print("\n💡 Key Observations:")
    print("   • Many weights are very small (close to zero)")
    print("   • Weight distribution typically: mean ≈ 0, concentrated near zero")
    print("   • Small weights contribute little to final predictions")
    print("   • Typical finding: 50-90% of weights can be removed!")

    print("\n🎯 The Problem:")
    print("   Why store and compute with weights that barely matter?")
    print("   • They take memory")
    print("   • They require computation")
    print("   • They slow down inference")
    print("   • But removing them has minimal accuracy impact!")

    print("\n✨ The Solution:")
    print("   Prune (remove) small weights:")
    print("   • Magnitude pruning: Set small weights to zero")
    print("   • Structured pruning: Remove entire neurons/channels")
    print("   • Typical: 80-90% sparsity with <1% accuracy loss")
    print("   • Benefit: Smaller models, faster inference, less memory\n")

if __name__ == "__main__":
    show_weight_distribution_motivation()

🧪 Profiling Parameter Distribution:

   Total parameters: 262,656
   Model memory: 1.0 MB (FP32)

   Weight Statistics:
   Mean: 0.0353
   Std:  0.0267
   Min:  0.0000
   Max:  0.1962

   Weights Below Threshold:
   Threshold  |  Percentage
   -----------|--------------
   < 0.001   |    1.8%
   < 0.01    |   17.9%
   < 0.1     |   97.6%
   < 0.5     |  100.0%

💡 Key Observations:
   • Many weights are very small (close to zero)
   • Weight distribution typically: mean ≈ 0, concentrated near zero
   • Small weights contribute little to final predictions
   • Typical finding: 50-90% of weights can be removed!

🎯 The Problem:
   Why store and compute with weights that barely matter?
   • They take memory
   • They require computation
   • They slow down inference
   • But removing them has minimal accuracy impact!

✨ The Solution:
   Prune (remove) small weights:
   • Magnitude pruning: Set small weights to zero
   • Structured pruning: Remove entire neurons/channels
   • Typical: 80-90%

In [3]:
#| export
def measure_sparsity(model) -> float:
    """
    Calculate the percentage of zero weights in a model.

    TODO: Count zero weights and total weights across all layers

    APPROACH:
    1. Iterate through all model parameters
    2. Count zeros using np.sum(weights == 0)
    3. Count total parameters
    4. Return percentage: zeros / total * 100

    Args:
        model: Model with .parameters() method

    Returns:
        Sparsity percentage (0.0-100.0)

    EXAMPLE:
    >>> # Create test model with explicit composition
    >>> layer1 = Linear(10, 5)
    >>> layer2 = Linear(5, 2)
    >>> model = Sequential(layer1, layer2)
    >>> sparsity = measure_sparsity(model)
    >>> print(f"Model sparsity: {sparsity:.1f}%")
    Model sparsity: 0.0%  # Before pruning

    HINT: Use np.sum() to count zeros efficiently
    """
    ### BEGIN SOLUTION
    total_params = 0
    zero_params = 0

    for param in model.parameters():
        # only count weight matrices (2D), not biases (1D)
        # biases are often initialized to zero, which would skew sparsity
        if len(param.shape) > 1:
            total_params += param.size
            zero_params += np.sum(param.data == 0)
    
    if total_params == 0:
        return 0.0
    
    return (zero_params / total_params) * 100.0
    ### END SOLUTION

In [4]:
def test_unit_measure_sparsity():
    """🧪 Test sparsity measurement functionality."""
    print("🧪 Unit Test: Measure Sparsity...")

    # Test with dense model - explicit composition shows structure
    layer1 = Linear(4, 3)
    layer2 = Linear(3, 2)
    model = Sequential(layer1, layer2)  # Test helper for parameter collection

    initial_sparsity = measure_sparsity(model)
    assert initial_sparsity < 1.0, f"Expected <1% sparsity (dense model), got {initial_sparsity}%"

    # Test with manually sparse model - students see which weights are zeroed
    layer1.weight.data[0, 0] = 0  # Zero out specific weight
    layer1.weight.data[1, 1] = 0  # Zero out another weight
    sparse_sparsity = measure_sparsity(model)
    assert sparse_sparsity > 0, f"Expected >0% sparsity, got {sparse_sparsity}%"

    print("✅ measure_sparsity works correctly!")

if __name__ == "__main__":
    test_unit_measure_sparsity()

🧪 Unit Test: Measure Sparsity...
✅ measure_sparsity works correctly!


In [5]:
#| export
def magnitude_prune(model, sparsity=0.9):
    """
    Remove weights with smallest magnitudes to achieve target sparsity.

    TODO: Implement global magnitude-based pruning

    APPROACH:
    1. Collect all weights from the model
    2. Calculate absolute values to get magnitudes
    3. Find threshold at desired sparsity percentile
    4. Set weights below threshold to zero (in-place)

    EXAMPLE:
    >>> # Create model with explicit layer composition
    >>> layer1 = Linear(100, 50)
    >>> layer2 = Linear(50, 10)
    >>> model = Sequential(layer1, layer2)
    >>> original_params = sum(p.size for p in model.parameters())
    >>> magnitude_prune(model, sparsity=0.8)
    >>> final_sparsity = measure_sparsity(model)
    >>> print(f"Achieved {final_sparsity:.1f}% sparsity")
    Achieved 80.0% sparsity

    HINTS:
    - Use np.percentile() to find threshold
    - Modify model parameters in-place
    - Consider only weight matrices, not biases
    """
    ### BEGIN SOLUTION
    # Collect all weights (excluding biases)
    all_weights = []
    weight_params = []

    for param in model.parameters(): # [weight, bias]
        # skip biases (typically 1D)
        if len(param.shape) > 1:
            all_weights.extend(param.data.flatten())
            weight_params.append(param)

    if not all_weights:
        return model
    
    # calculate magnitude threshold
    magnitudes = np.abs(all_weights)
    threshold = np.percentile(magnitudes, sparsity * 100)

    # apply pruning to each weight parameter
    for param in weight_params:
        mask = np.abs(param.data) >= threshold
        param.data = param.data * mask

    return model
    ### END SOLUTION

In [6]:
def test_unit_magnitude_prune():
    """🧪 Test magnitude-based pruning functionality."""
    print("🧪 Unit Test: Magnitude Prune...")

    # Create test model with explicit composition - students see structure
    layer1 = Linear(4, 3)
    layer2 = Linear(3, 2)
    model = Sequential(layer1, layer2)

    # Set specific weight values for predictable testing
    # Students can see exactly which weights we're testing
    layer1.weight.data = np.array([
        [1.0, 2.0, 3.0],    # Large weights - should survive pruning
        [0.1, 0.2, 0.3],    # Medium weights
        [4.0, 5.0, 6.0],    # Large weights - should survive pruning
        [0.01, 0.02, 0.03]  # Tiny weights - will be pruned
    ])

    initial_sparsity = measure_sparsity(model)
    assert initial_sparsity < 1.0, "Model should start with minimal sparsity (<1%)"

    # Apply 50% pruning - removes smallest 50% of weights
    magnitude_prune(model, sparsity=0.5)
    final_sparsity = measure_sparsity(model)

    # Should achieve approximately 50% sparsity
    assert 40 <= final_sparsity <= 60, f"Expected ~50% sparsity, got {final_sparsity}%"

    # Verify largest weights survived - students understand pruning criteria
    remaining_weights = layer1.weight.data[layer1.weight.data != 0]
    assert len(remaining_weights) > 0, "Some weights should remain"
    assert np.all(np.abs(remaining_weights) >= 0.1), "Large weights should survive"

    print("✅ magnitude_prune works correctly!")

if __name__ == "__main__":
    test_unit_magnitude_prune()

🧪 Unit Test: Magnitude Prune...
✅ magnitude_prune works correctly!
